### WealthForge QLoRA on a free T4

Base `Qwen/Qwen3.6-8B` · 74 SFT examples · 4-bit + LoRA r=16. Runtime ▸ Change type ▸ T4 (free). Expect roughly **4–5 min** for 3 epochs at batch 2×4, under 8 GB VRAM. Upload `dataset/` (zip is fine) to `/content` before running cell 3.

In [ ]:
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
!ls dataset | head -20 || echo 'no dataset/ — upload dataset.zip and: !unzip dataset.zip'

In [ ]:
#@title model + dataset
import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

BASE = "Qwen/Qwen3.6-8B"
DATASET_DIR = "dataset"
MAX_SEQ = 2048
CHAT_TEMPLATE = "chatml"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE,
    max_seq_length=MAX_SEQ,
    dtype=None,                       # bf16 on Ampere+, fp16 otherwise
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias="none",
    use_gradient_checkpointing="unsloth",     # Unsloth kernel; use True elsewhere
    random_state=3407,
)

model = get_chat_template(model, chat_template=CHAT_TEMPLATE)
tokenizer.chat_template = model.tokenizer.chat_template

# The corpus is ShareGPT `messages` rows (see dataset/dataset-card.md for provenance).
ds = load_dataset("json", data_files={
    "train": f"{DATASET_DIR}/sft-train.jsonl",
    "validation": f"{DATASET_DIR}/sft-val.jsonl",
})


def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                     for m in batch["messages"]]}


train_ds = ds["train"].map(to_text, batched=True, remove_columns=ds["train"].column_names)
val_ds = ds["validation"].map(to_text, batched=True, remove_columns=ds["validation"].column_names) \
    if "validation" in ds else None

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_length=MAX_SEQ,
    args=SFTConfig(
        output_dir="out/finetune/qlora",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3.0,
        learning_rate=2e-4,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=10,
        save_strategy="steps",
        save_steps=10,
        eval_strategy="steps" if val_ds is not None else "no",
        eval_steps=10,
        fp16=False,
        bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
        report_to="none",
    ),
)

# Train on the assistant spans only. Without this the model spends capacity learning to
# reproduce the prompt, which is the classic way a small fine-tune gets worse, not better.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
trainer.train()

### Save, then hand it back to the agent

The adapter is tens of MB. The base weights were never touched, so rollback is deleting a directory — that asymmetry is the whole reason LoRA is the default for a $0 operation.

In [ ]:
trainer.save_model('/content/adapter')
model.save_pretrained_merged('/content/merged', tokenizer, save_method='merged_16bit')
!zip -qr /content/adapter.zip /content/adapter

Then serve it (`ollama create wf-house -f training/Modelfile`, or llama.cpp `--lora`) and run `./wf eval --compare` **before** changing any default. A fine-tune that scores lower on constraint obedience than the base model is a regression, not progress.